In [0]:
import requests, json, os
from datetime import datetime, timedelta

BASE_PATH = "/Volumes/workspace/bronze/raw_files/weather"

# Por enquanto só São Paulo
CIDADE = {"nome": "sao_paulo", "lat": -23.55, "lon": -46.63}

agora = datetime.now()
ingestion_date = agora.strftime("%Y-%m-%d")
ts = agora.strftime("%H%M%S")


def salvar_json(tipo, payload):
    pasta = f"{BASE_PATH}/{tipo}/ingestion_date={ingestion_date}"
    os.makedirs(pasta, exist_ok=True)
    caminho = f"{pasta}/{CIDADE['nome']}_{ts}.json"
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)
    return caminho


# 1) Previsão (próximos 7 dias)
r = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": CIDADE["lat"],
        "longitude": CIDADE["lon"],
        "hourly": "temperature_2m,apparent_temperature,precipitation_probability,precipitation",
        "timezone": "America/Sao_Paulo",
        "forecast_days": 7,
    },
    timeout=30,
)
r.raise_for_status()
print("Forecast salvo em:", salvar_json("forecast", r.json()))


In [0]:
# 2) Histórico (últimos 30 dias, a API de arquivo tem ~2 dias de atraso)
fim = agora - timedelta(days=2)
inicio = fim - timedelta(days=30)

r = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": CIDADE["lat"],
        "longitude": CIDADE["lon"],
        "start_date": inicio.strftime("%Y-%m-%d"),
        "end_date": fim.strftime("%Y-%m-%d"),
        "hourly": "temperature_2m,apparent_temperature,precipitation",
        "timezone": "America/Sao_Paulo",
    },
    timeout=60,
)
r.raise_for_status()
print("Histórico salvo em:", salvar_json("historical", r.json()))

In [0]:
display(dbutils.fs.ls(f"{BASE_PATH}/forecast/ingestion_date={ingestion_date}"))
display(dbutils.fs.ls(f"{BASE_PATH}/historical/ingestion_date={ingestion_date}"))